# Diniz Ch.7 — Ten-sample normalized-lattice prediction trace

Problem 14：x[k]=sin(πk/4)，order=2、λ=0.99、epsilon=0.01。d=0 用來執行 prediction branch；此處檢查 ef_bar/delta_bar/eta，不把 joint-process 的 d=0 誤差當成 predictor performance。

來源：`Diniz-Chap7-SignalPred.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Signal and normalized-lattice trace

In [ ]:
x = np.sin(np.pi * np.arange(10) / 4)
batch = SampleBatch(x, np.zeros_like(x))
model = af.NormalizedLatticeRLS(filter_order=1, lam=0.99, epsilon=0.01)
result = training.run(model, batch, trace_keys=("ef_bar", "delta_bar", "eta_f", "eta_b"))

## 3. Print and show the prediction-branch state

In [ ]:
print("k       x        ef_bar[-1]       delta[0]       delta[1]       eta_f[0]       eta_b[0]")
for k in range(len(x)):
  state = result.extras
  print(f"{k:1d} {x[k]:9.5f} {state['ef_bar'][k, -1]:16.6f} "
        f"{state['delta_bar'][k, 0]:14.6f} {state['delta_bar'][k, 1]:14.6f} "
        f"{state['eta_f'][k, 0]:14.6f} {state['eta_b'][k, 0]:14.6f}")
plot.lines({
  "Input": x,
  "Normalized forward prediction error": result.extras["ef_bar"][:, -1]
}, title="Normalized prediction branch: ten samples")
plot.show()